# Homework: Inverted Pendulum World Model Training

This notebook runs the official homework workflow: install dependencies, generate MuJoCo ground-truth windows, train the starter world model, evaluate locked nMSE/VPT scoreboard metrics, and plot diagnostics.

The assignment trains **one dynamics world model**. It does not train a policy, does not use reward, and does not use actor-critic updates.

The public scoreboard is for debugging and comparison. Official grading uses TA-private data generated with hidden seeds.

## 1. Configure Repository

If you are running this notebook inside a local clone, the setup cell will reuse the current repository. Otherwise, set `COURSE_REPO_URL` to your fork and the notebook will clone it.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

COURSE_REPO_URL = "https://github.com/EriciusX/EEC289A_WorldModel-Homework.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path.cwd().resolve()
if not (COURSE_REPO_DIR / "wm_hw").exists() and (COURSE_REPO_DIR.parent / "wm_hw").exists():
    COURSE_REPO_DIR = COURSE_REPO_DIR.parent.resolve()
os.chdir(COURSE_REPO_DIR)
SMOKE_RUN = False
RUN_SCOREBOARD_DEMO = True
RUN_DATA_GENERATION = False
RUN_STARTER_TRAINING = False
FINAL_CHECKPOINT_DIR = Path("artifacts/final_exp_p_new/best_checkpoint")
FINAL_ARTIFACT_DIR = Path("artifacts/final_exp_p_new")
DATASET_DIR = Path("data/public_scoreboard")

print("COURSE_REPO_URL =", COURSE_REPO_URL)
print("COURSE_REPO_DIR =", COURSE_REPO_DIR)
print("FINAL_CHECKPOINT_DIR =", FINAL_CHECKPOINT_DIR)
print("DATASET_DIR =", DATASET_DIR)
print("checkpoint exists:", (FINAL_CHECKPOINT_DIR / "checkpoint.pt").exists())


COURSE_REPO_URL = https://github.com/EriciusX/EEC289A_WorldModel-Homework.git
COURSE_REPO_DIR = D:\Github_Libs\EEC289A_WorldModel-Homework
FINAL_CHECKPOINT_DIR = artifacts\final_exp_p_new\best_checkpoint
DATASET_DIR = data\public_scoreboard
checkpoint exists: True


## 2. Install Dependencies And Verify MuJoCo

In [2]:
# Dependencies are already installed in the local conda environment.
import gymnasium as gym
import torch

env = gym.make("InvertedPendulum-v5", reset_noise_scale=0.01)
obs, info = env.reset(seed=0)
print("obs shape:", obs.shape)
print("action shape:", env.action_space.shape)
print("action range:", env.action_space.low, env.action_space.high)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
env.close()


obs shape: (4,)
action shape: (1,)
action range: [-3.] [3.]
torch: 2.11.0+cu128
cuda available: True


## 3. Run Tests

These tests check the environment, dataset schema, no-leak open-loop rollout, differentiable loss, locked VPT/nMSE metrics, and train/eval smoke path.

In [3]:
!{sys.executable} -m pytest -q -m "not slow"


.............                                                            [100%]
13 passed, 1 deselected in 2.33s


## 4. Generate MuJoCo Ground-Truth Windows

In [4]:
from pathlib import Path
DATASET_DIR = Path("data/public_scoreboard")
required_splits = ["train", "val", "test", "ood"]
missing_splits = [split for split in required_splits if not (DATASET_DIR / f"{split}.npz").exists()]

if RUN_DATA_GENERATION or missing_splits:
    print("Generating MuJoCo windows into", DATASET_DIR)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "wm_hw.dataset",
            "--config",
            "configs/public_scoreboard.yaml",
            "--output-dir",
            str(DATASET_DIR),
        ],
        check=True,
    )
else:
    print("Using existing DATASET_DIR:", DATASET_DIR)

for split in required_splits:
    print(f"{split} exists:", (DATASET_DIR / f"{split}.npz").exists())


Using existing DATASET_DIR: data\public_scoreboard
train exists: True
val exists: True
test exists: True
ood exists: True


## 5. Train The Starter World Model

In [5]:
TRAIN_OUTPUT_DIR = Path("artifacts/colab_starter_train")

if RUN_STARTER_TRAINING:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "wm_hw.train",
            "--config",
            "configs/student.yaml",
            "--model",
            "student",
            "--dataset-dir",
            str(DATASET_DIR),
            "--output-dir",
            str(TRAIN_OUTPUT_DIR),
        ],
        check=True,
    )
    FINAL_CHECKPOINT_DIR = TRAIN_OUTPUT_DIR / "best_checkpoint"
    FINAL_ARTIFACT_DIR = TRAIN_OUTPUT_DIR
else:
    # Final submission path: use the selected checkpoint produced by the optimization run.
    FINAL_CHECKPOINT_DIR = Path("artifacts/final_exp_p_new/best_checkpoint")
    FINAL_ARTIFACT_DIR = Path("artifacts/final_exp_p_new")

print("Using final checkpoint:", FINAL_CHECKPOINT_DIR)


Using final checkpoint: artifacts\final_exp_p_new\best_checkpoint


## 6. Evaluate Test And OOD VPT Scoreboard

In [6]:
COLAB_TEST_EVAL_DIR = FINAL_ARTIFACT_DIR / "colab_eval_test"
COLAB_OOD_EVAL_DIR = FINAL_ARTIFACT_DIR / "colab_eval_ood"

for split, output_dir in [("test", COLAB_TEST_EVAL_DIR), ("ood", COLAB_OOD_EVAL_DIR)]:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "wm_hw.eval_horizon",
            "--checkpoint-dir",
            str(FINAL_CHECKPOINT_DIR),
            "--dataset-dir",
            str(DATASET_DIR),
            "--split",
            split,
            "--horizon",
            "auto",
            "--eval-config",
            "configs/official_eval.yaml",
            "--output-dir",
            str(output_dir),
        ],
        check=True,
    )

# Explicit notebook-visible summary for the submission output.
for split, output_dir in [("test", COLAB_TEST_EVAL_DIR), ("ood", COLAB_OOD_EVAL_DIR)]:
    with open(output_dir / "metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)
    print(f"{split} metrics")
    print("  VPT80@0.25:", metrics["VPT80@0.25"])
    print("  VPT50@0.25:", metrics["VPT50@0.25"])
    print("  nMSE@10:", metrics["nMSE@10"])
    print("  nMSE@100:", metrics["nMSE@100"])
    print("  nMSE@1000:", metrics["nMSE@1000"])
    print("  nMSE_AUC:", metrics["nMSE_AUC"])


test metrics
  VPT80@0.25: 29
  VPT50@0.25: 35
  nMSE@10: 0.0005431876634247601
  nMSE@100: 0.1510320007801056
  nMSE@1000: 0.2656916379928589
  nMSE_AUC: 0.20463311672210693
ood metrics
  VPT80@0.25: 29
  VPT50@0.25: 36
  nMSE@10: 0.0009689766447991133
  nMSE@100: 0.15268245339393616
  nMSE@1000: 0.27087604999542236
  nMSE_AUC: 0.21189261972904205


## 7. Plot Diagnostics

In [7]:
COLAB_PLOTS_DIR = FINAL_ARTIFACT_DIR / "colab_plots"
subprocess.run(
    [
        sys.executable,
        "-m",
        "wm_hw.plotting",
        "--eval-dir",
        str(COLAB_TEST_EVAL_DIR),
        "--output-dir",
        str(COLAB_PLOTS_DIR),
    ],
    check=True,
)


import json
with open(COLAB_TEST_EVAL_DIR / "metrics.json", "r", encoding="utf-8") as f:
    test_metrics = json.load(f)
with open(COLAB_OOD_EVAL_DIR / "metrics.json", "r", encoding="utf-8") as f:
    ood_metrics = json.load(f)

print("test VPT80@0.25:", test_metrics["VPT80@0.25"])
print("ood VPT80@0.25:", ood_metrics["VPT80@0.25"])
print("test max_horizon:", test_metrics["max_horizon"])
print("test nMSE_AUC:", test_metrics["nMSE_AUC"])
print("test nMSE@1000:", test_metrics["nMSE@1000"])
print("plots:", COLAB_PLOTS_DIR)


test VPT80@0.25: 29
ood VPT80@0.25: 29
test max_horizon: 1000
test nMSE_AUC: 0.20463311672210693
test nMSE@1000: 0.2656916379928589
plots: artifacts\final_exp_p_new\colab_plots


## 8. Optional Long-Horizon Scoreboard

The regular cells above stay lightweight. If you want the final public scoreboard artifacts, set `RUN_SCOREBOARD_DEMO = True` and `SMOKE_RUN = False` in the first cell. The public scoreboard config uses `warmup_steps=10` and `max_horizon=1000`.

In [8]:
# Final scoreboard packaging for selected checkpoint.
import json
import zipfile
from pathlib import Path

ARTIFACT_ROOT = Path("artifacts/final_exp_p_new")
with open(ARTIFACT_ROOT / "colab_eval_test" / "scoreboard_summary.json", "r", encoding="utf-8") as f:
    print(json.load(f))


{'VPT50@0.25': 35, 'VPT80@0.25': 29, 'VPT80_fraction': 0.029, 'checkpoint_step': 9800, 'max_horizon': 1000, 'model_name': 'student', 'nMSE@10': 0.0005431876634247601, 'nMSE@100': 0.1510320007801056, 'nMSE@1000': 0.2656916379928589, 'nMSE@90': 0.1457478404045105, 'nMSE_AUC': 0.20463311672210693, 'primary_metric': 'VPT80@0.25', 'step_nMSE@10': 0.0006991497939452529, 'step_nMSE@100': 0.1906062662601471, 'step_nMSE@1000': 0.44484513998031616}
